# EPIC Clarity Person Hydration

This notebook hydrates the OMOP PERSON table from EPIC Clarity patient master data.

## Source Tables
- `_exponent._bronze_epic_clarity_*.dbo_PATIENT` - Patient demographics
- `_exponent._bronze_epic_clarity_*.dbo_PATIENT_4` - Extended patient attributes (sex at birth, gender identity)
- `_exponent._bronze_epic_clarity_*.dbo_PATIENT_RACE` - Patient race information

## OMOP Fields Populated
- person_id (surrogate key)
- birth_datetime
- gender_concept_id (from gender_source_value)
- race_concept_id (from race_source_value)
- ethnicity_concept_id (from ethnicity_source_value)
- provider_id (current primary care provider)
- care_site_id (primary care location)
- location_id (residential location)

In [ ]:
source = 'epic_clarity'

In [ ]:
silver_person_df = spark.sql(f'''
SELECT
    CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', p.PAT_ID) AS person_source_value,
    p.BIRTH_DATE AS birth_datetime,
    YEAR(p.BIRTH_DATE) AS year_of_birth,
    MONTH(p.BIRTH_DATE) AS month_of_birth,
    DAY(p.BIRTH_DATE) AS day_of_birth,
    COALESCE(p4.SEX_ASGN_AT_BIRTH_C_NAME, '') AS gender_source_value,
    COALESCE(pr.PATIENT_RACE_C_NAME, '') AS race_source_value,
    COALESCE(p.ETHNIC_GROUP_C_NAME, '') AS ethnicity_source_value,
    CONCAT_WS(CHR(31), 'epic_clarity', 'CLARITY_SER', 'PROV_ID', p.CUR_PCP_PROV_ID_PROV_NAME) AS provider_source_value,
    CONCAT_WS(CHR(31), 'epic_clarity', 'CLARITY_DEP', 'DEPARTMENT_ID', p.CUR_PRIM_LOC_ID_LOC_NAME) AS care_site_source_value,
    CONCAT_WS(CHR(31), 'epic_clarity', 'PATIENT', 'PAT_ID', p.PAT_ID) AS location_source_value,
    CURRENT_TIMESTAMP() AS updated_tsp
FROM _exponent._bronze_epic_clarity.patient p
LEFT JOIN _exponent._bronze_epic_clarity.patient_3 p4 ON p.PAT_ID = p4.PAT_ID
LEFT JOIN _exponent._bronze_epic_clarity.patient_race pr ON p.PAT_ID = pr.PAT_ID
WHERE p.PAT_ID IS NOT NULL
''')

display(silver_person_df)
silver_person_df.createOrReplaceTempView("silver_person")

In [ ]:
%sql
MERGE INTO _exponent.omop_silver.person AS t
USING silver_person AS s
ON t.person_source_value = s.person_source_value

WHEN MATCHED AND (
     NOT (t.gender_concept_id <=> s.gender_concept_id)
  OR NOT (t.year_of_birth <=> s.year_of_birth)
  OR NOT (t.month_of_birth <=> s.month_of_birth)
  OR NOT (t.day_of_birth <=> s.day_of_birth)
  OR NOT (t.birth_datetime <=> s.birth_datetime)
  OR NOT (t.race_concept_id <=> s.race_concept_id)
  OR NOT (t.ethnicity_concept_id <=> s.ethnicity_concept_id)
  OR NOT (t.provider_source_value <=> s.provider_source_value)
  OR NOT (t.care_site_source_value <=> s.care_site_source_value)
  OR NOT (t.location_source_value <=> s.location_source_value)
)
THEN UPDATE SET
  t.birth_datetime = s.birth_datetime,
  t.year_of_birth = s.year_of_birth,
  t.month_of_birth = s.month_of_birth,
  t.day_of_birth = s.day_of_birth,
  t.gender_source_value = s.gender_source_value,
  t.race_source_value = s.race_source_value,
  t.ethnicity_source_value = s.ethnicity_source_value,
  t.provider_source_value = s.provider_source_value,
  t.care_site_source_value = s.care_site_source_value,
  t.location_source_value = s.location_source_value,
  t.updated_tsp = s.updated_tsp

WHEN NOT MATCHED THEN
INSERT (
  person_source_value,
  birth_datetime,
  year_of_birth,
  month_of_birth,
  day_of_birth,
  gender_source_value,
  race_source_value,
  ethnicity_source_value,
  provider_source_value,
  care_site_source_value,
  location_source_value,
  updated_tsp
)
VALUES (
  s.person_source_value,
  s.birth_datetime,
  s.year_of_birth,
  s.month_of_birth,
  s.day_of_birth,
  s.gender_source_value,
  s.race_source_value,
  s.ethnicity_source_value,
  s.provider_source_value,
  s.care_site_source_value,
  s.location_source_value,
  s.updated_tsp
)

In [ ]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_person (
    source_system,
    person_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    s.source_system,
    s.person_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    COALESCE(s.updated_tsp, CURRENT_TIMESTAMP()) AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT 'epic_clarity' AS source_system, person_source_value, updated_tsp
    FROM _exponent.omop_silver.person
    WHERE person_source_value IS NOT NULL
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_person x
  ON s.person_source_value = x.person_source_value
  AND x.source_system = 'epic_clarity'

In [ ]:
gold_person_df = spark.sql("""
SELECT
    source_to_person.person_id,
    s.birth_datetime,
    s.year_of_birth,
    s.month_of_birth,
    s.day_of_birth,
    COALESCE(gc.concept_id, 0) AS gender_concept_id,
    COALESCE(rc.concept_id, 0) AS race_concept_id,
    COALESCE(ec.concept_id, 0) AS ethnicity_concept_id,
    mp.provider_id,
    mc.care_site_id,
    ml.location_id,
    s.updated_tsp
FROM _exponent.omop_silver.person s
INNER JOIN _exponent.omop_mapping.source_to_person source_to_person
    ON s.person_source_value = source_to_person.person_source_value
    AND source_to_person.source_system = 'epic_clarity'
    AND source_to_person.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept gc
    ON gc.source_id = s.gender_source_value
    AND gc.domain_id = 'Gender'
    AND gc.source_system = 'epic_clarity'
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept rc
    ON rc.source_id = s.race_source_value
    AND rc.domain_id = 'Race'
    AND rc.source_system = 'epic_clarity'
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept ec
    ON ec.source_id = s.ethnicity_source_value
    AND ec.domain_id = 'Ethnicity'
    AND ec.source_system = 'epic_clarity'
LEFT JOIN _exponent.omop_mapping.source_to_provider mp
    ON s.provider_source_value = mp.provider_source_value
    AND mp.source_system = 'epic_clarity'
    AND mp.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.source_to_care_site mc
    ON s.care_site_source_value = mc.care_site_source_value
    AND mc.source_system = 'epic_clarity'
    AND mc.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.source_to_location ml
    ON s.location_source_value = ml.location_source_value
    AND ml.source_system = 'epic_clarity'
    AND ml.active_flag = TRUE
""")

display(gold_person_df)
gold_person_df.createOrReplaceTempView("gold_person")

In [ ]:
%sql
MERGE INTO _exponent.omop.person AS gold_person
USING gold_person AS src
ON gold_person.person_id = src.person_id

WHEN MATCHED AND (
     NOT (gold_person.gender_concept_id <=> src.gender_concept_id)
  OR NOT (gold_person.year_of_birth <=> src.year_of_birth)
  OR NOT (gold_person.month_of_birth <=> src.month_of_birth)
  OR NOT (gold_person.day_of_birth <=> src.day_of_birth)
  OR NOT (gold_person.birth_datetime <=> src.birth_datetime)
  OR NOT (gold_person.race_concept_id <=> src.race_concept_id)
  OR NOT (gold_person.ethnicity_concept_id <=> src.ethnicity_concept_id)
  OR NOT (gold_person.provider_id <=> src.provider_id)
  OR NOT (gold_person.care_site_id <=> src.care_site_id)
  OR NOT (gold_person.location_id <=> src.location_id)
)
THEN UPDATE SET
  gold_person.birth_datetime = src.birth_datetime,
  gold_person.year_of_birth = src.year_of_birth,
  gold_person.month_of_birth = src.month_of_birth,
  gold_person.day_of_birth = src.day_of_birth,
  gold_person.gender_concept_id = src.gender_concept_id,
  gold_person.race_concept_id = src.race_concept_id,
  gold_person.ethnicity_concept_id = src.ethnicity_concept_id,
  gold_person.provider_id = src.provider_id,
  gold_person.care_site_id = src.care_site_id,
  gold_person.location_id = src.location_id

WHEN NOT MATCHED THEN INSERT (
  person_id,
  birth_datetime,
  year_of_birth,
  month_of_birth,
  day_of_birth,
  gender_concept_id,
  race_concept_id,
  ethnicity_concept_id,
  provider_id,
  care_site_id,
  location_id
)
VALUES (
  src.person_id,
  src.birth_datetime,
  src.year_of_birth,
  src.month_of_birth,
  src.day_of_birth,
  src.gender_concept_id,
  src.race_concept_id,
  src.ethnicity_concept_id,
  src.provider_id,
  src.care_site_id,
  src.location_id
)